# Python Bridge for 12b

## A Short Code Refresher

Use this optional notebook alongside 12b to inspect a fitted smoothing model, distinguish fitted predictions from updated levels, and issue forecasts on future dates. Allow about 20 minutes. A six-observation SES example and a constructed quarterly series provide all the data.

The lecture explains the component updates. Here we connect those updates to constructor settings, a fitted result object, its attributes, and the table returned by an error comparison.

## References

The statsmodels [exponential-smoothing examples](https://www.statsmodels.org/stable/examples/notebooks/generated/exponential_smoothing.html) demonstrate the fitting and forecasting interfaces. [FPP Pythonic §§8.1–8.3](https://otexts.com/fpppy/08-exponential-smoothing.html) explains the component methods. The [course explorer quickstart](https://github.com/olearydj/INSY7130/blob/main/lectures/12b-exponential-smoothing/12b-forecast-explorer-quickstart.md) describes interactive forecast/observe/update experiments.

## Inspect a Known SES Update

Use the six observations 40, 44, 42, 46, 48, and 45. Fix the starting level at 40 and alpha at 0.3 to make the updates easy to check. `initialization_method="known"` and `optimized=False` make those supplied choices explicit instead of estimating them.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display
from statsmodels.tsa.holtwinters import ExponentialSmoothing, SimpleExpSmoothing

observations = pd.Series([40.0, 44.0, 42.0, 46.0, 48.0, 45.0], name="Observed")
ses = SimpleExpSmoothing(observations, initialization_method="known", initial_level=40.0).fit(
    smoothing_level=0.3, optimized=False
)
updates = pd.DataFrame(
    {
        "Observed": observations,
        "Prediction before observation": ses.fittedvalues,
        "Level after observation": ses.level,
    }
)
display(updates.round(3))

Before observing 44, the prediction is 40. After observing 44, the level becomes 0.3 × 44 + 0.7 × 40 = 41.2. That updated level predicts the following observation. A fitted prediction and the level updated with the same observation are different quantities.

## Forecast Beyond the Supplied Observations

`.forecast(3)` asks for three future observations. SES has no slope or seasonal state to extend, so all three point forecasts use the final level.

In [ ]:
future_ses = ses.forecast(3)
display(future_ses.rename("Future forecast"))
print("Final level:", ses.level.iloc[-1])
assert np.allclose(future_ses, ses.level.iloc[-1])

The future row labels continue after the six supplied observations. The forecast does not update itself with unavailable future measurements.

## Prepare a Seasonal Training Series

Construct eight years of quarterly values. A rising level is multiplied by seasonal factors 0.8, 1.0, 1.25, and 0.95, with small independent simulated disturbances added. The fixed random seed reproduces those disturbances. Reserve the final eight quarters for evaluation.

In [ ]:
rng = np.random.default_rng(7130)
dates = pd.period_range("2018Q1", periods=32, freq="Q")
levels = 100 + 1.2 * np.arange(32)
factors = np.tile([0.8, 1.0, 1.25, 0.95], 8)
series = pd.Series(levels * factors + rng.normal(0, 1, 32), index=dates, name="Observed")
training, test = series.iloc[:-8], series.iloc[-8:]
assert training.index.max() < test.index.min()
display(training.tail())

The `PeriodIndex` gives the model quarterly spacing. `seasonal_periods=4` below means four observations per annual cycle; it is unrelated to the eight-quarter forecast horizon.

## Separate Model Settings From Fitted Parameters

Choose additive trend, multiplicative seasonality, and damping in the constructor. Multiplicative seasonality fits the known proportional pattern here and requires positive observations. Calling `.fit()` estimates the smoothing weights and initial states from the training data, including the damping parameter phi.

In [ ]:
model = ExponentialSmoothing(
    training,
    trend="add",
    damped_trend=True,
    seasonal="mul",
    seasonal_periods=4,
    initialization_method="estimated",
)
result = model.fit()
parameter_names = ["smoothing_level", "smoothing_trend", "smoothing_seasonal", "damping_trend"]
display(pd.Series({name: result.params[name] for name in parameter_names}).round(4))

The constructor determines which components exist. The parameter dictionary reports the weights the fit selected. A small seasonal smoothing weight would mean little updating of the seasonal states, not the absence of a seasonal pattern.

## Inspect the Fitted States and Errors

The fitted result stores a level, slope, seasonal factor, and one-step fitted prediction for each training observation. Put them on the training index so one row shows the values associated with one quarter.

In [ ]:
states = pd.DataFrame(
    {
        "Observed": training,
        "Fitted prediction": result.fittedvalues,
        "Updated level": result.level,
        "Updated slope": result.trend,
        "Updated seasonal factor": result.season,
    }
)
states["Fitted error"] = states["Observed"] - states["Fitted prediction"]
display(states.tail().round(3))

The fitted error compares an observation with its prediction before that observation's component update. Slope is in units per quarter; seasonality is a factor. Read their scales separately. The initial states and parameters were estimated using the complete training history, so these are fitted errors rather than an untouched evaluation period.

## Issue and Evaluate a Fixed-Origin Forecast

Forecast eight quarters from the training endpoint. Compare with a seasonal-naive reference that repeats the last four training observations twice. Check the future dates, then calculate errors against the same withheld values.

In [ ]:
forecast = result.forecast(len(test))
seasonal_naive = pd.Series(np.tile(training.iloc[-4:].to_numpy(), 2), index=test.index)
predictions = pd.DataFrame({"Smoothing model": forecast, "Seasonal naive": seasonal_naive})
assert predictions.index.equals(test.index)
assert predictions.notna().all().all()
errors = predictions.rsub(test, axis="index")
scores = pd.DataFrame(
    {"MAE (units)": errors.abs().mean(), "RMSE (units)": np.sqrt(errors.pow(2).mean())}
)
display(pd.DataFrame({"Observed": test}).join(predictions).round(3))
display(scores.round(3))

The table measures one forecast origin on a constructed example. The model choices match the pattern we planted, so their performance does not establish a generally best specification. The lecture's broader comparison needs the same information boundary and an appropriate reference forecast.

Plot the training series, withheld observations, and both forecasts on a common scale.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4), layout="constrained")
ax.plot(training.index.to_timestamp(), training, "o-", color="0.4", label="Training")
ax.plot(test.index.to_timestamp(), test, "o-", color="black", label="Withheld observations")
for name in predictions:
    ax.plot(test.index.to_timestamp(), predictions[name], "--", label=name)
ax.axvline(test.index[0].start_time, color="0.5", linestyle=":")
ax.set(title="Forecasts From the Training Endpoint", xlabel="Quarter start", ylabel="Units")
ax.legend()
plt.show()

## Try One Small Modification

Fit another SES result with alpha 0.8 while keeping the six observations and starting level unchanged. Before running the cell, predict whether the updated levels will respond more or less strongly to each new observation.

In [ ]:
responsive = SimpleExpSmoothing(
    observations, initialization_method="known", initial_level=40.0
).fit(smoothing_level=0.8, optimized=False)
display(
    pd.DataFrame(
        {
            "Observed": observations,
            "Level, alpha 0.3": ses.level,
            "Level, alpha 0.8": responsive.level,
        }
    ).round(3)
)

The larger weight brings each updated level closer to its new observation. It changes the fitted update sequence and final level, while future SES forecasts still remain flat.

## Ready for 12b

Return to the lecture when you can distinguish model structure from fitted parameters, inspect the state arrays, separate fitted predictions from updated states, and evaluate a future forecast against an explicit baseline.

---

Auburn University / Industrial and Systems Engineering<br>
INSY 7130, Pattern Discovery and Time Series Analysis<br>
© Copyright Danny J. O'Leary.

For course materials, attribution, and licensing information, see the [INSY 7130 course-materials README](../../README.md).